### Import Dependencies

In [ ]:
import openai
from qdrant_client import QdrantClient

### Embedding Function

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding

### Retrieval Function

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01", query=query_embedding, limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
      if not result.payload: raise ValueError("No payload found in Qdrant ScoredPoint")
      retrieved_context_ids.append(result.payload["parent_asin"])  
      retrieved_context.append(result.payload["preprocessed_description"])
      similarity_scores.append(result.score)
      retrieved_context_ratings.append(result.payload["average_rating"])


    return {
      "retrieved_context_ids": retrieved_context_ids,
      "retrieved_context": retrieved_context,
      "similarity_scores": similarity_scores,
      "retrieved_context_ratings": retrieved_context_ratings
    }

In [ ]:
retrieved_context = retrieve_data("do you have anything that can help me attach my apple pencil to my ipad?", k=10)

In [ ]:
retrieved_context

### Format retrieved context function

In [ ]:
def process_context(context):
  formatted_context = ""

  for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
    formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

  return formatted_context

In [ ]:
preprocessed_context = process_context(retrieved_context)

In [ ]:
print(preprocessed_context)

### Create prompt template function

In [ ]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt

In [ ]:
prompt = build_prompt(preprocessed_context, "do you have anything that can help me attach my apple pencil to my ipad?")

In [ ]:
print(prompt)

### Generate answer function

In [ ]:
def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt},
        ],
        reasoning_effort="none",
    )

    return response.choices[0].message.content

In [ ]:
print(generate_answer(prompt))

Combined RAG pipeline

In [ ]:
def rag_pipeline(question, topk_k=5):

  qdrant_client = QdrantClient(url="http://localhost:6333")
  retrieved_context = retrieve_data(question, k=topk_k)
  preprocessed_context = process_context(retrieved_context)
  prompt = build_prompt(preprocessed_context, question)
  answer = generate_answer(prompt)
  return answer

In [ ]:
print(rag_pipeline("Got any cool earphones? I am only interested in the ones rated 4.5 or lower"))